In [2]:
import requests
import pandas as pd
import time
import io
from typing import List, Dict, Any, Tuple
import json
from tqdm import tqdm

import openmeteo_requests
import requests_cache
from retry_requests import retry

import cdsapi
import xarray as xr

In [2]:
district_id_map = {
    "002": "24 Parganas(S)",
    "001": "24 Parganas(N)",
    "003": "Bankura",
    "005": "Burdwan",
    "004": "Birbhum",
    "027": "Burdwan(E)",
    "026": "Burdwan(W)",
    "006": "Coochbehar",
    "007": "Darjeeling",
    "008": "Dinajpur(N)",
    "009": "Dinajpur(S)",
    "010": "Hooghly",
    "011": "Howrah",
    "012": "Jalpaiguri",
    "021": "Jhargram",
    "022": "Kalimpong",
    "013": "Kolkata",
    "014": "Maldah",
    "015": "Medinipore(E)",
    "016": "Medinipore(W)",
    "017": "Murshidabad",
    "018": "Nadia",
    "025": "Others",
    "019": "Purulia"
    }
df_dist_id_maps = pd.DataFrame(district_id_map, index=[0])
df_dist_id_maps.to_csv("../datasets/id_dist_mapping.csv",encoding= 'utf_8', index=False)

In [3]:
main_url = "http://emis.wbpcb.gov.in/airquality/JSP/aq/filter_for_aqiM.jsp"
ajax_url = "http://emis.wbpcb.gov.in/airquality/JSP/aq/fetch_val_ajax.jsp" 

In [ ]:
from json import JSONDecodeError


def station_grabber(dist_id: str, main_url: str, ajax_url: str) -> Dict[str, str]:
    all_stations : Dict[str, str] = {}

    with requests.Session() as session:
        print("Grabbing session cookies...")
        session.get(main_url, headers={"User-Agent": "Mozilla/5.0"})
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:149.0) Gecko/20100101 Firefox/149.0',
            'Accept': 'application/json, text/javascript, */*; q=0.01',
            'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
            'X-Requested-With': 'XMLHttpRequest',
            'Origin': 'http://emis.wbpcb.gov.in',
            'Referer': main_url
        }
        
        print(f"Fetching station maps for district_{dist_id}...")
        
        payload = {
            'id': dist_id,
            'type': 'stationM'
        }
        
        response = session.post(ajax_url, data=payload, headers= headers)
        try:
            data = response.json()
            station_list = data.get("list", [])
            for stn in station_list:
                all_stations[stn['name']] = stn['id']
        except JSONDecodeError:
            print(f"  [!] Failed to parse district {dist_id}. Server said: {response.text[:100]}")
            
    time.sleep(0.5) #wanna be polite to the server 🙏🏿
    return all_stations

station_datas : Dict[str, Any] = {}
for dist in district_id_map.keys():
    stn_under_dist = station_grabber(dist, main_url=main_url, ajax_url=ajax_url)
    station_datas[dist] = stn_under_dist

In [5]:
all_station_list_vanilla = [x for dict in station_datas.values() for x in dict.values()]

In [ ]:
def date_availability_of_station(stn_code: str, main_url: str, ajax_url: str) -> List:
    all_available_dates : List = []

    with requests.Session() as session:
        print("Grabbing session cookies...")
        session.get(main_url, headers={"User-Agent": "Mozilla/5.0"})
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:149.0) Gecko/20100101 Firefox/149.0',
            'Accept': 'application/json, text/javascript, */*; q=0.01',
            'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
            'X-Requested-With': 'XMLHttpRequest',
            'Origin': 'http://emis.wbpcb.gov.in',
            'Referer': main_url
        }
        
        print(f"Fetching station maps for district_{stn_code}...")
        
        payload = {
            'stn_code': stn_code,
            'type': 'date'
        }
        
        response = session.post(ajax_url, data=payload, headers= headers)
        
        try:
            data = response.json()
            date_list = data.get("list", [])
            
            all_available_dates = [list(x.values())[0] for x in date_list]
        except JSONDecodeError:
            print(f"  [!] Failed to parse district {stn_code}. Server said: {response.text[:100]}")
            
    time.sleep(0.5) #wanna be polite to the server 🙏🏿
    return all_available_dates

#print(date_availability_of_station("st30", main_url, ajax_url))
all_available_dates : dict[str, List] = {}
for id in all_station_list_vanilla:
    all_available_dates[id] = date_availability_of_station(id,main_url, ajax_url)

In [ ]:
def date_availability_of_station(stn_code: str, date: str, main_url: str, ajax_url: str) -> Dict[str, Any]:
    aqi_dict : Dict[str, Any] = {} 

    with requests.Session() as session:
        print("Grabbing session cookies...")
        session.get(main_url, headers={"User-Agent": "Mozilla/5.0"})
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:149.0) Gecko/20100101 Firefox/149.0',
            'Accept': 'application/json, text/javascript, */*; q=0.01',
            'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
            'X-Requested-With': 'XMLHttpRequest',
            'Origin': 'http://emis.wbpcb.gov.in',
            'Referer': main_url
        }
        
        print(f"Fetching station maps for district_{stn_code}...")
        
        payload = {
            'stn_code': stn_code,
            'date': date,
            'type': "aqi"
        }
        
        response = session.post(ajax_url, data=payload, headers= headers)
        
        try:
            data = response.json()
            aqi_data_specific = data.get("list", [])

            for param_dict in aqi_data_specific:
                aqi_dict[param_dict['pname']] = float(param_dict['value'])
        except JSONDecodeError:
            print(f"  [!] Failed to parse district {stn_code}. Server said: {response.text[:100]}")
            
    #time.sleep(0.5) #wanna be polite to the server 🙏🏿
    return aqi_dict

date_availability_of_station('st67', "25/02/2026", main_url, ajax_url)

In [ ]:
def process_single_station(stn_code:str, date_list:List[str]) -> Tuple[str, List[Dict[str, Any]]]:
    station_results: List[Dict[str, Any]] = []
    
    
    with requests.Session() as session:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:149.0) Gecko/20100101 Firefox/149.0',
            'Accept': 'application/json, text/javascript, */*; q=0.01',
            'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
            'X-Requested-With': 'XMLHttpRequest',
            'Origin': 'http://emis.wbpcb.gov.in',
            'Referer': "http://emis.wbpcb.gov.in/airquality/JSP/aq/filter_for_aqiM.jsp"
        }
        
        session.get("http://emis.wbpcb.gov.in/airquality/JSP/aq/filter_for_aqiM.jsp", headers={"User-Agent": "Mozilla/5.0"})
        
        for date_str in tqdm(date_list, desc=f"Processing for station {stn_code}"):
            aqi_dict: Dict[str, Any] = {}
            
            m, d, y = date_str.split('/') 
            date_m = f"{d}/{m}/{y}"
            payload = {
                    'stn_code': stn_code,
                    'date': date_m,
                    'type': "aqi"
                }
            response = session.post("http://emis.wbpcb.gov.in/airquality/JSP/aq/fetch_val_ajax.jsp" , data=payload, headers= headers)
            try:
                
                data = response.json()
                aqi_data_specific = data.get("list", [])

                for param_dict in aqi_data_specific:
                    aqi_dict[param_dict['pname']] = float(param_dict['value'])
            except JSONDecodeError:
                print(f"  [!] Failed to parse district {stn_code}. Server said: {response.text[:100]}")
            
            aqi_dict['date'] = date_m 
            station_results.append(aqi_dict)
    
    return stn_code, station_results

In [ ]:
aqi_data_acquired: Dict[str, List[Dict[str, Any]]] = {}

for stn, date_list in all_available_dates.items():
    _, station_results = process_single_station(stn, date_list)
    aqi_data_acquired[stn] = station_results

In [ ]:
with open("../datasets/aqi_data.json", "w") as file:
    json.dump(aqi_data_acquired, file)

In [ ]:
refactored_data : List[Dict[str, Any]] = []

for stn_code, list_data in aqi_data_acquired.items():

    district : str = ""
    location : str = ""
    for dist_c, stn_dict in station_datas.items():
        for loc, stn_c in stn_dict.items():
            if stn_code == stn_c:
                district = dist_c
                location = loc

    element = {
        "stn_code": stn_code,
        "district": district_id_map[district],
        "location": location,
        "data": list_data
    }
    
    refactored_data.append(element)

In [18]:
with open("../datasets/AQMS_Manual_stations_datasets/aqi_data.json", "w") as file:
    json.dump(refactored_data, file)

In [3]:
with open("../datasets/AQMS_Manual_stations_datasets/aqi_data.json", "r") as file:
    data = json.load(file)

In [2]:
availability_array : Dict[str, List[Any]]
with open("../datasets/AQMS_datasets/aqms_availability_report_final.json", "r") as file:
    availability_array = json.load(file)

In [49]:
def geocode_single_location(location:str) -> None|Dict[str, float]:
    url = "https://api.olamaps.io/places/v1/geocode"
    params = {
        "address": location,
        "language": "English",
        "api_key": "h5sR18hSupJaBPJrX87BvE2Nhaycp9gvCfPtbgx6"
    }
    headers = {
        "X-Request-Id": "",
        "X-Correlation-Id": "",
        "Accept": "application/json"
    }
    try:
        response = requests.get(url, params=params, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        print(f"Success for {location}")
        return data['geocodingResults'][0]['geometry']['location']
    except requests.exceptions.RequestException as e:
        print(f"API request failed {e} for location {location}")

In [ ]:
for items in availability_array['Sheet1']:
    location = items['location'] + ', '+ items['district']
    geometry = geocode_single_location(location=location)
    if geometry:
        items['lng'] = geometry['lng']
        items['lat'] = geometry['lat']
    else:
        items['lng'] = None 
        items['lat'] = None 

In [58]:
with open("../datasets/AQMS_datasets/aqms_availability_report_final.json", "w") as file:
    json.dump(availability_array, file)

In [4]:
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [ ]:
url = "https://archive-api.open-meteo.com/v1/archive"
for items in availability_array['Sheet1']:
    item_lat = items['lat']
    item_lng = items['lng']
    params = {
		"latitude": item_lat,
		"longitude": item_lng,
		"start_date": "2025-06-30",
		"end_date": "2025-10-30",
		"hourly": ["wind_speed_10m", "wind_direction_10m", "wind_speed_100m", "wind_direction_100m"],
		"timezone": "auto",
	}
    responses = openmeteo.weather_api(url, params = params)
    response = responses[0]
    print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
    print(f"Elevation: {response.Elevation()} m asl")
    print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
    print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
    
    hourly = response.Hourly()
    hourly_wind_speed_10m = hourly.Variables(0).ValuesAsNumpy()
    hourly_wind_direction_10m = hourly.Variables(1).ValuesAsNumpy()
    hourly_wind_speed_100m = hourly.Variables(2).ValuesAsNumpy()
    hourly_wind_direction_100m = hourly.Variables(3).ValuesAsNumpy()
    
    hourly_data = {
		"date": pd.date_range(
			start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
			end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
			freq = pd.Timedelta(seconds = hourly.Interval()),
			inclusive = "left"
		).tz_convert("Asia/Kolkata")
	}
    
    hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
    hourly_data["wind_direction_10m"] = hourly_wind_direction_10m
    hourly_data["wind_speed_100m"] = hourly_wind_speed_100m
    hourly_data["wind_direction_100m"] = hourly_wind_direction_100m
    
    hourly_dataframe = pd.DataFrame(data = hourly_data)
    
    item_file = items['filenumber']
    hourly_dataframe.to_csv(f"../datasets/AQMS_datasets/meteo-data/{item_file}.csv")

In [ ]:
dataset = "derived-era5-single-levels-daily-statistics"
request = {
    "product_type": "reanalysis",
    "year": "2025",
    "month": [
        "07", "08", "09",
        "10"
    ],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "daily_statistic": "daily_maximum",
    "time_zone": "utc+00:00",
    "frequency": "1_hourly",
    "variable": ["boundary_layer_height"],
    "area": [23.8, 86.9, 23.3, 87.6]
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()

In [3]:
ds = xr.open_dataset("../datasets/BLH_datasets/34af4e4324f5397750952f527196f439.nc")
point = ds.sel(
    latitude=23.55,
    longitude=87.32,
    method="nearest"
)

In [4]:
df = point.to_dataframe()

print(df.head())

                   blh  number  latitude  longitude
valid_time                                         
2025-07-01  316.747070       0      23.5      87.25
2025-07-02  190.338455       0      23.5      87.25
2025-07-03   44.086288       0      23.5      87.25
2025-07-04   62.548859       0      23.5      87.25
2025-07-05   70.910599       0      23.5      87.25


In [5]:
df.to_csv("../datasets/BLH_datasets/durgapur_data_min.csv")